In [58]:
import math
import os
import time
import contextlib
from collections import deque
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Iterable, Iterator, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem
from rdkit import RDLogger
from rdkit.Chem import rdchem
from torch import distributed as dist
from torch.utils.data import DataLoader, Dataset, DistributedSampler, Sampler

RDLogger.DisableLog("rdApp.*")

In [59]:
def _build_mapping(values: Iterable[int]) -> Dict[int, int]:
    """
    Creates a hashmap within which keys correspond to features and values correspond to integer encodings

    Args:
        values (Iterable[int]): atom/bond, i.e., node/edge features to encode

    Returns:
        Dict[int, int]: mapping between atom/ bond (node/ edge features) and their corresponding encodings
    """
    return {value: idx for idx, value in enumerate(values)}

def _one_hot(value: any, mapping: Dict[any,int]) -> np.ndarray[float]:
    """
    Create a one-hot encoded vector of floats (0.0) wherein the index of the value corresponding to the input feature reads 1.0.
    This index is given by the numerical encoding of the input feature and this numerical encoding is in turn given by an input hashmap.

    Args:
        value (any): input feature (e.g., atom type, atom degree, hybridization state)
        mapping (Dict[any:int]): a hashmap connecting the input feature to its numerical encoding

    Returns:
        np.ndarray: a vector of zeros (0.0, float) wherein the index corresponding to the numerical encoding of the input feature has been switched to 1.0 (float)
    """
    size = len(mapping) + 1 
    vec = np.zeros(size, dtype=np.float32) # use floats rather than integers for neural network training
    vec[mapping.get(value, len(mapping))] = 1.0 # flip the bit corresponding to the mapping index to 1
    return vec


In [60]:
### ------ atom features used to create node matrix ------

# for the following atom features, we create a hashmap to get numerical encodings for each input feature
# these numerical encodings will later be converted to one-hot encoded vectors per feature
# and all one-hot-encoded features will subsequently be concatenated along the row dimension to give a single feature vector per atom/ node

ATOM_TYPES = [1, 5, 6, 7, 8, 9, 14, 15, 16, 17, 35, 53] # [H, B, C, N, O, F, Si, P, S, Cl, Br, I]
ATOM_MAP = _build_mapping(ATOM_TYPES)

DEGREES = [0, 1, 2, 3, 4, 5]
DEGREE_MAP = _build_mapping(DEGREES)

FORMAL_CHARGES = [-2,1,0,1,2]
CHARGE_MAP = _build_mapping(FORMAL_CHARGES)

NUM_HS = [0, 1, 2, 3, 4]
NUM_H_MAP = _build_mapping(NUM_HS)

HYBRIDIZATIONS = [rdchem.HybridizationType.SP,
                  rdchem.HybridizationType.SP2,
                  rdchem.HybridizationType.SP3,
                  rdchem.HybridizationType.SP3D,
                  rdchem.HybridizationType.SP3D2]
HYB_MAP = _build_mapping(HYBRIDIZATIONS)

NODE_FEAT_DIM: int = len(ATOM_MAP) + len(DEGREE_MAP) + len(CHARGE_MAP) + len(NUM_H_MAP) + len(HYB_MAP)
print(f"\nLength of node feature vector: {NODE_FEAT_DIM}\n")

### ------ bond features used to create edge matrix ------

# for the following edge features, we similarly create a hashmap to get numerical encodings

BOND_TYPES = [rdchem.BondType.SINGLE,
              rdchem.BondType.DOUBLE,
              rdchem.BondType.TRIPLE,
              rdchem.BondType.AROMATIC]
BOND_MAP = _build_mapping(BOND_TYPES)

EDGE_FEAT_DIM: int = len(BOND_TYPES) + 1 # we add a plus one to store any NONE bond types
print(f"\nlength of edge feature vector: {EDGE_FEAT_DIM}")


Length of node feature vector: 32


length of edge feature vector: 5


In [61]:
# one-hot encoding for atomic number of carbon (hence value = 6)
_one_hot(value = 6, mapping = ATOM_MAP)

array([0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.], dtype=float32)

In [62]:
def atom_to_feature(atom: rdchem.Atom) -> np.ndarray[float]:
    """
    Generates one-hot encoded vectors for each atom feature for a given RDKit rdchem.Atom object.
    These one-hot encoded feature vectors are then concatenated along the row-dimension to output a single row vector.

    Args:
        atom (rdchem.Atom): RDKit atom object.

    Returns:
        np.ndarray[float]: Single output vector formed by concatenating each one-hot-encoded feature vector along the row dimension.
    """
    feats = [
        _one_hot(atom.GetAtomicNum(), ATOM_MAP),
        _one_hot(atom.GetTotalDegree(), DEGREE_MAP),
        _one_hot(atom.GetFormalCharge(), CHARGE_MAP),
        _one_hot(atom.GetTotalNumHs(includeNeighbors=True), NUM_H_MAP),
        _one_hot(atom.GetHybridization(), HYB_MAP),
        np.array([atom.GetIsAromatic()], dtype=np.float32),
        np.array([atom.IsInRing()], dtype=np.float32),
    ]
    return np.concatenate(feats, axis=0)

In [63]:
mol = Chem.MolFromSmiles("CO")
for atom in mol.GetAtoms():
    print(f"Encoding for {atom.GetSymbol()} atom:")
    print(atom_to_feature(atom))
    print('')

Encoding for C atom:
[0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0.
 0. 0. 0. 0. 1. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]

Encoding for O atom:
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0.
 0. 0. 1. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]



In [64]:
def bond_to_feature(bond: rdchem.Bond) -> np.ndarray[float]:
    """
    Create a one-hot-encoded vector representing a given bond type.
    Bond types accounted for include single, double, triple, and aromatic.

    Args:
        bond (rdchem.Bond): RDKit bond object

    Returns:
        np.ndarray[float]: one-hot-encoded vector representing the bond type
    """
    vec = np.zeros(EDGE_FEAT_DIM, dtype=np.float32)
    if bond is None:
        vec[-1] = 1.0
    else:
        vec[BOND_MAP.get(bond.GetBondType(), EDGE_FEAT_DIM - 1)] = 1.0
        return vec

In [75]:
mol = Chem.MolFromSmiles("CO")
for bond in mol.GetBonds():
    u, v = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
    print(u,v)

0 1
